# Garmin health UI → API mapping

This notebook maps every stat shown in the **Health / Garmin** section of the frontend to the
`python-garminconnect` methods demonstrated in [`scripts/garmin_demo.py`](../../scripts/garmin_demo.py).

Run the cells below (after auth) to see which API returns each field and sample values for a chosen date.

**UI sources:** `HealthDayView`, `HealthWeekView`, `HealthMonthView`, `HealthActivityList` · **type:** `DailyHealth`

## Stats shown in the UI

### Day view
| Section | Fields |
|---------|--------|
| Steps | `steps`, `stepGoal` |
| Sleep | `sleepHours`, `deepSleepHours`, `sleepScore` |
| Calories | `activeCalories`, `totalCalories`, `bmrCalories` |
| Metric cards | `restingHr`, `hrv`, `hrvStatus`, `avgStress`, `maxStress`, `bodyBatteryLow/High`, `bodyBatteryCharged/Drained` |
| Spark charts | `bodyBatteryCurve`, `stressCurve` |
| Activities | per-activity `name`, `type`, `startTime`, `durationMin`, `calories`, `avgHr`, `distanceKm` |

### Week view (aggregated from daily data)
`avgSteps`, `avgSleepHours`, `avgActiveCalories`, `avgStress`, `avgHrv`, `totalActivities`, `totalWorkoutMin`, bar charts for steps/sleep/stress, weekly `moderateIntensityMin` / `vigorousIntensityMin` totals.

### Month view
`totalSteps`, `avgSteps`, `avgSleepHours`, `totalActivities`, `moderateIntensityMin` / `vigorousIntensityMin` totals, calendar cell `steps`, `activityByType` breakdown.

## API mapping (primary source per UI field)

Menu keys from `garmin_demo.py` → `Garmin` method → response path → UI field.

| UI field | `garmin_demo` key | API call | Response path |
|----------|-------------------|----------|---------------|
| `steps` | `get_user_summary` | `api.get_user_summary(date)` | `totalSteps` |
| `stepGoal` | `get_steps_data` | `api.get_steps_data(date)` | `dailyStepGoal` (dict) or step goal from goals/profile |
| `activeCalories` | `get_user_summary` | `api.get_user_summary(date)` | `activeKilocalories` |
| `totalCalories` | `get_user_summary` | `api.get_user_summary(date)` | `totalKilocalories` |
| `bmrCalories` | `get_user_summary` | `api.get_user_summary(date)` | `bmrKilocalories` |
| `moderateIntensityMin` | `get_user_summary` / `get_intensity_minutes_data` | `api.get_user_summary(date)` | `moderateIntensityMinutes` |
| `vigorousIntensityMin` | `get_user_summary` / `get_intensity_minutes_data` | `api.get_user_summary(date)` | `vigorousIntensityMinutes` |
| `restingHr` | `get_resting_heart_rate` / `get_heart_rates` | `api.get_heart_rates(date)` or `api.get_rhr_day(date)` | `restingHeartRate` |
| `sleepHours` | `get_sleep_data` | `api.get_sleep_data(date)` | `dailySleepDTO.sleepTimeSeconds` ÷ 3600 |
| `deepSleepHours` | `get_sleep_data` | `api.get_sleep_data(date)` | `dailySleepDTO.deepSleepSeconds` ÷ 3600 |
| `sleepScore` | `get_sleep_data` | `api.get_sleep_data(date)` | `dailySleepDTO.sleepScores.overall.value` |
| `hrv` | `get_hrv_data` | `api.get_hrv_data(date)` | `hrvSummary.lastNightAvg` |
| `hrvStatus` | `get_hrv_data` | `api.get_hrv_data(date)` | `hrvSummary.status` → map to balanced/low/high |
| `avgStress` | `get_all_day_stress` | `api.get_all_day_stress(date)` | `avgStressLevel` |
| `maxStress` | `get_all_day_stress` | `api.get_all_day_stress(date)` | `maxStressLevel` |
| `stressCurve` | `get_all_day_stress` | `api.get_all_day_stress(date)` | `stressValuesArray` (time series) |
| `bodyBatteryHigh` | `get_user_summary` / `get_body_battery` | `api.get_user_summary(date)` | `bodyBatteryHighestValue` |
| `bodyBatteryLow` | `get_user_summary` / `get_body_battery` | `api.get_user_summary(date)` | `bodyBatteryLowestValue` |
| `bodyBatteryCharged` | `get_body_battery` | `api.get_body_battery(date, date)[0]` | `charged` |
| `bodyBatteryDrained` | `get_body_battery` | `api.get_body_battery(date, date)[0]` | `drained` |
| `bodyBatteryCurve` | `get_body_battery` | `api.get_body_battery(date, date)[0]` | `bodyBatteryValuesArray` → `[epoch_ms, level]` |
| Activities list | `get_activities_by_date` | `api.get_activities_by_date(date, date)` | list of activity dicts |
| activity `name` | — | — | `activityName` |
| activity `type` | — | — | `activityType.typeKey` |
| activity `startTime` | — | — | `startTimeLocal` |
| activity `durationMin` | — | — | `duration` ÷ 60 |
| activity `calories` | — | — | `calories` |
| activity `avgHr` | — | — | `avgHR` or `averageHR` |
| activity `distanceKm` | — | — | `distance` ÷ 1000 |

**Bulk / range calls** (week/month views can use these instead of N× daily calls):

| Use | `garmin_demo` key | API call |
|-----|-------------------|----------|
| Daily steps range | `get_daily_steps` | `api.get_daily_steps(start, end)` |
| Body battery range | `get_body_battery` | `api.get_body_battery(start, end)` |
| Weekly intensity | `get_weekly_intensity_minutes` | `api.get_weekly_intensity_minutes(start, end)` |
| Activities in range | `get_activities_by_date` | `api.get_activities_by_date(start, end)` |
| Recent activities | `get_activities` | `api.get_activities(start, limit)` |

Notes:
- `get_stats` and `get_user_summary` are aliases. `get_stress_data` and `get_all_day_stress` hit the same daily stress endpoint.
- **`get_activities_fordate` is misleading** — despite the name and `garmin_demo` menu label, it calls `/mobile-gateway/heartRate/forDate` and returns `ActivitiesForDay`, `AllDayHR`, `SleepTimes` (HR chart overlays), **not** workout summaries. Use `get_activities_by_date` for the activities list.

In [19]:
import contextlib
import json
import logging
import os
import sys
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path
from typing import Any

from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

logging.getLogger("garminconnect").setLevel(logging.CRITICAL)

In [20]:
def safe_api_call(api_method, *args, **kwargs):
    """Call an API method; return (success, result, error_message)."""
    try:
        return True, api_method(*args, **kwargs), None
    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication error: {e}"
    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit: {e}"
    except GarminConnectConnectionError as e:
        error_str = str(e)
        if "400" in error_str:
            return False, None, "Not available (400)"
        if "404" in error_str:
            return False, None, "Not found (404)"
        return False, None, f"Connection error: {e}"
    except Exception as e:
        return False, None, f"Unexpected error: {e}"

In [21]:
def init_api() -> Garmin | None:
    """Restore saved tokens or log in. Tokens: ~/.garminconnect by default."""
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = str(Path(tokenstore).expanduser())

    try:
        garmin = Garmin()
        garmin.login(tokenstore_path)
        print(f"Logged in using saved tokens ({tokenstore_path}).")
        return garmin
    except GarminConnectTooManyRequestsError as err:
        print(f"Rate limit: {err}")
        return None
    except (GarminConnectAuthenticationError, GarminConnectConnectionError):
        print("No valid tokens — please log in.")

    while True:
        try:
            email = os.getenv("GARMIN_EMAIL") or os.getenv("EMAIL") or input("Email: ").strip()
            password = os.getenv("GARMIN_PASSWORD") or os.getenv("PASSWORD") or getpass("Password: ")
            garmin = Garmin(
                email=email,
                password=password,
                prompt_mfa=lambda: input("MFA code: ").strip(),
            )
            garmin.login(tokenstore_path)
            print(f"Login OK. Tokens saved to {tokenstore_path}")
            return garmin
        except GarminConnectAuthenticationError:
            print("Wrong credentials — try again.")
        except GarminConnectConnectionError as err:
            print(f"Connection error: {err}")
            return None
        except KeyboardInterrupt:
            return None

In [22]:
# Change this to probe a specific day
TARGET_DATE = date.today().isoformat()
WEEK_START = (date.fromisoformat(TARGET_DATE) - timedelta(days=6)).isoformat()

api = init_api()
if not api:
    raise SystemExit("Could not authenticate")

Logged in using saved tokens (/Users/andrewallkin/.garminconnect).


In [23]:
def dig(obj: Any, *keys, default=None):
    """Safely traverse nested dicts."""
    for key in keys:
        if not isinstance(obj, dict):
            return default
        obj = obj.get(key)
    return obj if obj is not None else default


def seconds_to_hours(seconds: int | float | None) -> float | None:
    if seconds is None:
        return None
    return round(seconds / 3600, 1)


def map_hrv_status(raw: str | None) -> str:
    """Map Garmin HRV status to UI HrvStatus."""
    if not raw:
        return "unavailable"
    raw = raw.upper()
    if raw in {"LOW", "POOR", "UNBALANCED"}:
        return "low"
    if raw in {"HIGH"}:
        return "high"
    if raw in {"BALANCED"}:
        return "balanced"
    return "unavailable"


def series_values(array: list | None, index: int = 1) -> list[int]:
    """Extract Y values from Garmin time-series arrays like [[epoch, value], ...]."""
    if not array:
        return []
    return [int(row[index]) for row in array if isinstance(row, (list, tuple)) and len(row) > index]


def preview(obj: Any, max_chars: int = 1200) -> str:
    text = json.dumps(obj, indent=2, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "\n... (truncated)"


def normalize_activity_list(raw: Any) -> list[dict]:
    """Normalize activity payloads from get_activities / get_activities_by_date."""
    if isinstance(raw, list):
        return [a for a in raw if isinstance(a, dict)]
    if isinstance(raw, dict):
        for key in ("activityList", "activities", "ActivitiesForDay"):
            value = raw.get(key)
            if isinstance(value, list):
                return [a for a in value if isinstance(a, dict)]
    return []


def activity_to_ui(act: dict) -> dict:
    """Map a Garmin activity dict to HealthActivity-shaped fields."""
    start_local = act.get("startTimeLocal") or ""
    start_time = start_local.split(" ")[1][:5] if " " in start_local else start_local[:5]
    duration = act.get("duration")
    distance = act.get("distance")
    return {
        "id": str(act.get("activityId")),
        "name": act.get("activityName"),
        "type": dig(act, "activityType", "typeKey") or "other",
        "startTime": start_time,
        "durationMin": round(duration / 60) if duration else 0,
        "calories": act.get("calories") or 0,
        "avgHr": act.get("avgHR") or act.get("averageHR") or 0,
        "distanceKm": round(distance / 1000, 2) if distance else None,
    }


def fetch_activities_for_date(api: Garmin, date_str: str) -> list[dict]:
    """Fetch recorded workouts for one calendar day."""
    ok, raw, _ = safe_api_call(api.get_activities_by_date, date_str, date_str)
    return normalize_activity_list(raw) if ok else []

## 1. Daily summary — `get_user_summary` (menu: Daily Health → option 2)

Covers most activity totals: steps, calories, intensity minutes, and body battery high/low on the daily summary.

In [24]:
ok, summary, err = safe_api_call(api.get_user_summary, TARGET_DATE)
print(f"api.get_user_summary('{TARGET_DATE}') → {'OK' if ok else err}")

if ok and summary:
    ui_fields = {
        "steps": summary.get("totalSteps"),
        "activeCalories": summary.get("activeKilocalories"),
        "totalCalories": summary.get("totalKilocalories"),
        "bmrCalories": summary.get("bmrKilocalories"),
        "moderateIntensityMin": summary.get("moderateIntensityMinutes"),
        "vigorousIntensityMin": summary.get("vigorousIntensityMinutes"),
        "bodyBatteryHigh": summary.get("bodyBatteryHighestValue"),
        "bodyBatteryLow": summary.get("bodyBatteryLowestValue"),
        "bodyBatteryCharged (alt)": summary.get("bodyBatteryChargedValue"),
        "bodyBatteryDrained (alt)": summary.get("bodyBatteryDrainedValue"),
    }
    print("\nUI fields from user summary:")
    for k, v in ui_fields.items():
        print(f"  {k}: {v}")
    print("\nAll keys:", sorted(summary.keys()))

api.get_user_summary('2026-06-22') → OK

UI fields from user summary:
  steps: 4152
  activeCalories: 557.0
  totalCalories: 2462.0
  bmrCalories: 1905.0
  moderateIntensityMin: 50
  vigorousIntensityMin: 22
  bodyBatteryHigh: 92
  bodyBatteryLow: 20
  bodyBatteryCharged (alt): 76
  bodyBatteryDrained (alt): 63

All keys: ['abnormalHeartRateAlertsCount', 'activeKilocalories', 'activeSeconds', 'activityStressDuration', 'activityStressPercentage', 'averageMonitoringEnvironmentAltitude', 'averageSpo2', 'averageStressLevel', 'avgWakingRespirationValue', 'bmrKilocalories', 'bodyBatteryActivityEventList', 'bodyBatteryAtWakeTime', 'bodyBatteryChargedValue', 'bodyBatteryDrainedValue', 'bodyBatteryDuringSleep', 'bodyBatteryDynamicFeedbackEvent', 'bodyBatteryHighestValue', 'bodyBatteryLowestValue', 'bodyBatteryMostRecentValue', 'bodyBatteryVersion', 'burnedKilocalories', 'calendarDate', 'consumedKilocalories', 'dailyStepGoal', 'durationInMilliseconds', 'endOfDayBodyBatteryDynamicFeedbackEvent', 

## 2. Step goal — `get_steps_data` (menu: Daily Health → option 4)

In [25]:
ok, steps_data, err = safe_api_call(api.get_steps_data, TARGET_DATE)
print(f"api.get_steps_data('{TARGET_DATE}') → {'OK' if ok else err}")

if ok:
    if isinstance(steps_data, dict):
        print(f"  stepGoal (dailyStepGoal): {steps_data.get('dailyStepGoal')}")
        print(f"  totalSteps: {steps_data.get('totalSteps')}")
        print("  keys:", sorted(steps_data.keys()))
    elif isinstance(steps_data, list):
        print(f"  returned list with {len(steps_data)} entries (hourly step chart)")
        if steps_data:
            print("  first entry keys:", steps_data[0].keys() if isinstance(steps_data[0], dict) else steps_data[0])
    else:
        print(preview(steps_data))

api.get_steps_data('2026-06-22') → OK
  returned list with 84 entries (hourly step chart)
  first entry keys: dict_keys(['startGMT', 'endGMT', 'steps', 'pushes', 'primaryActivityLevel', 'activityLevelConstant'])


## 3. Sleep — `get_sleep_data` (menu: Daily Health → option 7)

In [26]:
ok, sleep, err = safe_api_call(api.get_sleep_data, TARGET_DATE)
print(f"api.get_sleep_data('{TARGET_DATE}') → {'OK' if ok else err}")

if ok and sleep:
    dto = dig(sleep, "dailySleepDTO") or {}
    ui_fields = {
        "sleepHours": seconds_to_hours(dto.get("sleepTimeSeconds")),
        "deepSleepHours": seconds_to_hours(dto.get("deepSleepSeconds")),
        "sleepScore": dig(dto, "sleepScores", "overall", "value"),
    }
    print("\nUI fields from sleep data:")
    for k, v in ui_fields.items():
        print(f"  {k}: {v}")
    print("\ndailySleepDTO keys:", sorted(dto.keys()) if dto else "(none)")
    print("\nTop-level keys:", sorted(sleep.keys()))

api.get_sleep_data('2026-06-22') → OK

UI fields from sleep data:
  sleepHours: 7.5
  deepSleepHours: 0.8
  sleepScore: 85

dailySleepDTO keys: ['ageGroup', 'autoSleepEndTimestampGMT', 'autoSleepStartTimestampGMT', 'averageRespirationValue', 'averageSpO2HRSleep', 'averageSpO2Value', 'avgHeartRate', 'avgSleepStress', 'awakeCount', 'awakeSleepSeconds', 'breathingDisruptionSeverity', 'calendarDate', 'deepSleepSeconds', 'deviceRemCapable', 'highestRespirationValue', 'highestSpO2Value', 'id', 'lightSleepSeconds', 'lowestRespirationValue', 'lowestSpO2Value', 'napTimeSeconds', 'nextSleepNeed', 'remSleepSeconds', 'retro', 'sleepEndTimestampGMT', 'sleepEndTimestampLocal', 'sleepFromDevice', 'sleepNeed', 'sleepQualityTypePK', 'sleepResultTypePK', 'sleepScoreFeedback', 'sleepScoreInsight', 'sleepScorePersonalizedInsight', 'sleepScores', 'sleepStartTimestampGMT', 'sleepStartTimestampLocal', 'sleepTimeSeconds', 'sleepVersion', 'sleepWindowConfirmationType', 'sleepWindowConfirmed', 'unmeasurableSlee

## 4. Resting HR — `get_heart_rates` / `get_rhr_day` (menu: Daily Health → options 5 & 6)

In [27]:
ok, hr, err = safe_api_call(api.get_heart_rates, TARGET_DATE)
print(f"api.get_heart_rates('{TARGET_DATE}') → {'OK' if ok else err}")
if ok and hr:
    print(f"  restingHr: {hr.get('restingHeartRate')}")
    print(f"  keys: {sorted(hr.keys())}")

ok2, rhr, err2 = safe_api_call(api.get_rhr_day, TARGET_DATE)
print(f"\napi.get_rhr_day('{TARGET_DATE}') → {'OK' if ok2 else err2}")
if ok2 and rhr:
    print(preview(rhr, 800))

api.get_heart_rates('2026-06-22') → OK
  restingHr: 50
  keys: ['calendarDate', 'endTimestampGMT', 'endTimestampLocal', 'heartRateValueDescriptors', 'heartRateValues', 'lastSevenDaysAvgRestingHeartRate', 'maxHeartRate', 'minHeartRate', 'restingHeartRate', 'startTimestampGMT', 'startTimestampLocal', 'userProfilePK']

api.get_rhr_day('2026-06-22') → OK
{
  "userProfileId": 71924392,
  "statisticsStartDate": "2026-06-22",
  "statisticsEndDate": "2026-06-22",
  "allMetrics": {
    "metricsMap": {
      "WELLNESS_RESTING_HEART_RATE": [
        {
          "value": 50.0,
          "calendarDate": "2026-06-22"
        }
      ]
    }
  },
  "groupedMetrics": null
}


## 5. HRV — `get_hrv_data` (menu: Advanced Health → option 7)

In [28]:
ok, hrv, err = safe_api_call(api.get_hrv_data, TARGET_DATE)
print(f"api.get_hrv_data('{TARGET_DATE}') → {'OK' if ok else err}")

if ok and hrv:
    summary = dig(hrv, "hrvSummary") or {}
    raw_status = summary.get("status")
    ui_fields = {
        "hrv": summary.get("lastNightAvg"),
        "hrvStatus (raw)": raw_status,
        "hrvStatus (UI)": map_hrv_status(raw_status),
        "weeklyAvg": summary.get("weeklyAvg"),
        "baselineLow": summary.get("baselineBalancedLow"),
        "baselineHigh": summary.get("baselineBalancedHigh"),
    }
    print("\nUI fields from HRV:")
    for k, v in ui_fields.items():
        print(f"  {k}: {v}")
    print("\nhrvSummary keys:", sorted(summary.keys()) if summary else "(none)")

api.get_hrv_data('2026-06-22') → OK

UI fields from HRV:
  hrv: 90
  hrvStatus (raw): LOW
  hrvStatus (UI): low
  weeklyAvg: 77
  baselineLow: None
  baselineHigh: None

hrvSummary keys: ['baseline', 'calendarDate', 'createTimeStamp', 'feedbackPhrase', 'lastNight5MinHigh', 'lastNightAvg', 'status', 'weeklyAvg']


## 6. Stress — `get_all_day_stress` (menu: Daily Health → option 8)

Same endpoint as `get_stress_data` (Advanced Health → option 9). Provides averages plus `stressValuesArray` for the spark chart.

In [29]:
ok, stress, err = safe_api_call(api.get_all_day_stress, TARGET_DATE)
print(f"api.get_all_day_stress('{TARGET_DATE}') → {'OK' if ok else err}")

if ok and stress:
    curve = series_values(stress.get("stressValuesArray"))
    ui_fields = {
        "avgStress": stress.get("avgStressLevel"),
        "maxStress": stress.get("maxStressLevel"),
        "stressCurve samples": len(curve),
        "stressCurve first 8": curve[:8],
    }
    print("\nUI fields from stress:")
    for k, v in ui_fields.items():
        print(f"  {k}: {v}")
    print("\nkeys:", sorted(stress.keys()))

api.get_all_day_stress('2026-06-22') → OK

UI fields from stress:
  avgStress: 22
  maxStress: 90
  stressCurve samples: 419
  stressCurve first 8: [-1, 15, 14, 15, 18, 10, 8, 9]

keys: ['avgStressLevel', 'bodyBatteryValueDescriptorsDTOList', 'bodyBatteryValuesArray', 'calendarDate', 'endTimestampGMT', 'endTimestampLocal', 'maxStressLevel', 'startTimestampGMT', 'startTimestampLocal', 'stressChartValueOffset', 'stressChartYAxisOrigin', 'stressValueDescriptorsDTOList', 'stressValuesArray', 'userProfilePK']


## 7. Body battery — `get_body_battery` (menu: Historical → option 2)

Returns one dict per day in the range with `charged`, `drained`, and `bodyBatteryValuesArray` for the curve.

In [30]:
ok, bb_days, err = safe_api_call(api.get_body_battery, TARGET_DATE, TARGET_DATE)
print(f"api.get_body_battery('{TARGET_DATE}', '{TARGET_DATE}') → {'OK' if ok else err}")

if ok and bb_days:
    day_bb = bb_days[0] if isinstance(bb_days, list) and bb_days else {}
    curve = series_values(day_bb.get("bodyBatteryValuesArray"))
    ui_fields = {
        "bodyBatteryCharged": day_bb.get("charged"),
        "bodyBatteryDrained": day_bb.get("drained"),
        "bodyBatteryHigh (from series)": max(curve) if curve else None,
        "bodyBatteryLow (from series)": min(curve) if curve else None,
        "bodyBatteryCurve samples": len(curve),
        "bodyBatteryCurve first 8": curve[:8],
    }
    print("\nUI fields from body battery:")
    for k, v in ui_fields.items():
        print(f"  {k}: {v}")
    print("\nday keys:", sorted(day_bb.keys()) if isinstance(day_bb, dict) else type(day_bb))

api.get_body_battery('2026-06-22', '2026-06-22') → OK

UI fields from body battery:
  bodyBatteryCharged: 76
  bodyBatteryDrained: 63
  bodyBatteryHigh (from series): 92
  bodyBatteryLow (from series): 20
  bodyBatteryCurve samples: 6
  bodyBatteryCurve first 8: [20, 92, 85, 51, 53, 33]

day keys: ['bodyBatteryActivityEvent', 'bodyBatteryDynamicFeedbackEvent', 'bodyBatteryValueDescriptorDTOList', 'bodyBatteryValuesArray', 'charged', 'date', 'drained', 'endOfDayBodyBatteryDynamicFeedbackEvent', 'endTimestampGMT', 'endTimestampLocal', 'startTimestampGMT', 'startTimestampLocal']


## 8. Intensity minutes (detail) — `get_intensity_minutes_data` (menu: Advanced Health → option a)

In [31]:
ok, intensity, err = safe_api_call(api.get_intensity_minutes_data, TARGET_DATE)
print(f"api.get_intensity_minutes_data('{TARGET_DATE}') → {'OK' if ok else err}")
if ok and intensity:
    print(preview(intensity, 1000))

api.get_intensity_minutes_data('2026-06-22') → OK
{
  "userProfilePK": 71924392,
  "calendarDate": "2026-06-22",
  "startTimestampGMT": "2026-06-21T22:00:00.0",
  "endTimestampGMT": "2026-06-22T18:56:00.0",
  "startTimestampLocal": "2026-06-22T00:00:00.0",
  "endTimestampLocal": "2026-06-22T20:56:00.0",
  "weeklyModerate": 50,
  "weeklyVigorous": 22,
  "weeklyTotal": 94,
  "weekGoal": 300,
  "dayOfGoalMet": null,
  "startDayMinutes": 0,
  "endDayMinutes": 94,
  "moderateMinutes": 50,
  "vigorousMinutes": 22,
  "imValueDescriptorsDTOList": [
    {
      "index": 0,
      "key": "timestamp"
    },
    {
      "index": 1,
      "key": "value"
    }
  ],
  "imValuesArray": [
    [
      1782104399999,
      11
    ],
    [
      1782105299999,
      19
    ],
    [
      1782106199999,
      23
    ],
    [
      1782107099999,
      19
    ],
    [
      1782107999999,
      21
    ],
    [
      1782108899999,
      1
    ]
  ]
}


## 9. Activities — `get_activities_by_date` (menu: Activities → option by date range)

Use this for workout lists (strength, runs, etc.). **Do not use `get_activities_fordate`** — that method is misnamed and returns daily heart-rate data (`ActivitiesForDay`, `AllDayHR`, `SleepTimes`).

In [32]:
activities = fetch_activities_for_date(api, TARGET_DATE)
print(f"api.get_activities_by_date('{TARGET_DATE}', '{TARGET_DATE}') → {len(activities)} activities")

for act in activities[:5]:
    print(json.dumps(activity_to_ui(act), indent=2))

# Contrast with the misleading fordate endpoint (HR overlay, not workout list):
ok, hr_day, err = safe_api_call(api.get_activities_fordate, TARGET_DATE)
if ok and isinstance(hr_day, dict):
    print(f"\n(get_activities_fordate keys: {sorted(hr_day.keys())} — not used for activity list)")

api.get_activities_by_date('2026-06-22', '2026-06-22') → 1 activities
{
  "id": "23335691679",
  "name": "Strength",
  "type": "strength_training",
  "startTime": "06:47",
  "durationMin": 73,
  "calories": 565.0,
  "avgHr": 122.0,
  "distanceKm": null
}

(get_activities_fordate keys: ['ActivitiesForDay', 'AllDayHR', 'SleepTimes'] — not used for activity list)


## 10. Build a `DailyHealth`-shaped dict (all calls combined)

Minimal fetch set for the day view: `get_user_summary`, `get_steps_data`, `get_sleep_data`, `get_heart_rates`, `get_hrv_data`, `get_all_day_stress`, `get_body_battery`, **`get_activities_by_date`**.

For week/month aggregation, loop over dates or use range endpoints (`get_daily_steps`, `get_body_battery`, `get_activities_by_date`).

In [33]:
def build_daily_health(api: Garmin, date_str: str) -> dict:
    """Fetch Garmin data and shape it like frontend DailyHealth (best-effort)."""
    _, summary, _ = safe_api_call(api.get_user_summary, date_str)
    _, steps_data, _ = safe_api_call(api.get_steps_data, date_str)
    _, sleep, _ = safe_api_call(api.get_sleep_data, date_str)
    _, hr, _ = safe_api_call(api.get_heart_rates, date_str)
    _, hrv, _ = safe_api_call(api.get_hrv_data, date_str)
    _, stress, _ = safe_api_call(api.get_all_day_stress, date_str)
    _, bb_days, _ = safe_api_call(api.get_body_battery, date_str, date_str)
    act_list = fetch_activities_for_date(api, date_str)

    summary = summary or {}
    dto = dig(sleep, "dailySleepDTO") or {}
    hrv_summary = dig(hrv, "hrvSummary") or {}
    bb_day = (bb_days or [{}])[0] if isinstance(bb_days, list) else {}
    stress_curve = series_values((stress or {}).get("stressValuesArray"))
    bb_curve = series_values(bb_day.get("bodyBatteryValuesArray"))

    step_goal = summary.get("dailyStepGoal")
    if step_goal is None and isinstance(steps_data, dict):
        step_goal = steps_data.get("dailyStepGoal")

    activities = [activity_to_ui(act) for act in act_list]

    return {
        "date": date_str,
        "steps": summary.get("totalSteps") or 0,
        "stepGoal": step_goal or 10000,
        "totalCalories": summary.get("totalKilocalories") or 0,
        "activeCalories": summary.get("activeKilocalories") or 0,
        "bmrCalories": summary.get("bmrKilocalories") or 0,
        "restingHr": (hr or {}).get("restingHeartRate") or 0,
        "sleepHours": seconds_to_hours(dto.get("sleepTimeSeconds")) or 0,
        "deepSleepHours": seconds_to_hours(dto.get("deepSleepSeconds")) or 0,
        "sleepScore": dig(dto, "sleepScores", "overall", "value"),
        "hrv": hrv_summary.get("lastNightAvg"),
        "hrvStatus": map_hrv_status(hrv_summary.get("status")),
        "avgStress": (stress or {}).get("avgStressLevel") or 0,
        "maxStress": (stress or {}).get("maxStressLevel") or 0,
        "bodyBatteryHigh": summary.get("bodyBatteryHighestValue") or (max(bb_curve) if bb_curve else 0),
        "bodyBatteryLow": summary.get("bodyBatteryLowestValue") or (min(bb_curve) if bb_curve else 0),
        "bodyBatteryCharged": bb_day.get("charged") or summary.get("bodyBatteryChargedValue") or 0,
        "bodyBatteryDrained": bb_day.get("drained") or summary.get("bodyBatteryDrainedValue") or 0,
        "moderateIntensityMin": summary.get("moderateIntensityMinutes") or 0,
        "vigorousIntensityMin": summary.get("vigorousIntensityMinutes") or 0,
        "stressCurve": stress_curve,
        "bodyBatteryCurve": bb_curve,
        "activities": activities,
    }


daily = build_daily_health(api, TARGET_DATE)
print(json.dumps({k: v for k, v in daily.items() if k not in {"stressCurve", "bodyBatteryCurve"}}, indent=2))
print(f"stressCurve: {len(daily['stressCurve'])} points, bodyBatteryCurve: {len(daily['bodyBatteryCurve'])} points")

{
  "date": "2026-06-22",
  "steps": 4152,
  "stepGoal": 7000,
  "totalCalories": 2462.0,
  "activeCalories": 557.0,
  "bmrCalories": 1905.0,
  "restingHr": 50,
  "sleepHours": 7.5,
  "deepSleepHours": 0.8,
  "sleepScore": 85,
  "hrv": 90,
  "hrvStatus": "low",
  "avgStress": 22,
  "maxStress": 90,
  "bodyBatteryHigh": 92,
  "bodyBatteryLow": 20,
  "bodyBatteryCharged": 76,
  "bodyBatteryDrained": 63,
  "moderateIntensityMin": 50,
  "vigorousIntensityMin": 22,
  "activities": [
    {
      "id": "23335691679",
      "name": "Strength",
      "type": "strength_training",
      "startTime": "06:47",
      "durationMin": 73,
      "calories": 565.0,
      "avgHr": 122.0,
      "distanceKm": null
    }
  ]
}
stressCurve: 419 points, bodyBatteryCurve: 6 points


## 11. Week range helpers (for `HealthWeekView` / `HealthMonthView`)

In [34]:
ok, daily_steps, err = safe_api_call(api.get_daily_steps, WEEK_START, TARGET_DATE)
print(f"api.get_daily_steps('{WEEK_START}', '{TARGET_DATE}') → {'OK' if ok else err}")
if ok and daily_steps:
    print(f"  {len(daily_steps)} days returned")
    if daily_steps:
        print("  sample keys:", sorted(daily_steps[0].keys()) if isinstance(daily_steps[0], dict) else daily_steps[0])

ok, weekly_intensity, err = safe_api_call(api.get_weekly_intensity_minutes, WEEK_START, TARGET_DATE)
print(f"\napi.get_weekly_intensity_minutes('{WEEK_START}', '{TARGET_DATE}') → {'OK' if ok else err}")
if ok and weekly_intensity:
    print(preview(weekly_intensity, 800))

ok, range_acts, err = safe_api_call(api.get_activities_by_date, WEEK_START, TARGET_DATE)
print(f"\napi.get_activities_by_date('{WEEK_START}', '{TARGET_DATE}') → {'OK' if ok else err}")
if ok and range_acts:
    print(f"  {len(range_acts)} activities in range")

api.get_daily_steps('2026-06-16', '2026-06-22') → OK
  7 days returned
  sample keys: ['calendarDate', 'stepGoal', 'totalDistance', 'totalSteps']

api.get_weekly_intensity_minutes('2026-06-16', '2026-06-22') → OK
[
  {
    "calendarDate": "2026-06-15",
    "weeklyGoal": 300,
    "moderateValue": 243,
    "vigorousValue": 314
  },
  {
    "calendarDate": "2026-06-22",
    "weeklyGoal": 300,
    "moderateValue": 50,
    "vigorousValue": 22
  }
]

api.get_activities_by_date('2026-06-16', '2026-06-22') → OK
  8 activities in range
